In [ ]:
# NOT USING THIS LOL, just kept the code in case i need to copy something from here
# NOT
# USING
# THIS

import torch
import torch.nn as nn
import numpy as np
import scipy.io

# --- Load from MATLAB ---
mat = scipy.io.loadmat('heston_params.mat')
Param = mat['Param'].flatten()
V0, ThetaV, Kappa, SigmaV, RhoSV = Param
r = -0.001

# --- Estimate alpha, beta from TSLA data ---
# (from rolling mu/V regression as discussed)
alpha = ...
beta  = ...
gamma = 3.0  # risk aversion

# --- ANN Policy ---
class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

policy = PolicyNet()
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

# --- Training Loop ---
dt      = 1/252
T       = 3.0          # years
n_steps = int(T/dt)
n_paths = 1000

for epoch in range(2000):

    # --- Simulate V(t) paths ---
    V = torch.full((n_paths,), V0)
    W_ann      = torch.ones(n_paths)
    W_analytic = torch.ones(n_paths)

    for t in range(n_steps):

        # 1. Heston variance step
        dW_V = torch.randn(n_paths) * (dt**0.5)
        V = torch.clamp(
            V + Kappa*(ThetaV - V)*dt + SigmaV*torch.sqrt(V.clamp(min=0))*dW_V,
            min=1e-6
        )

        # 2. mu from V  ← both methods use this
        mu_t = alpha + beta * V

        # 3. shared wealth shock
        dW_S = torch.randn(n_paths) * (dt**0.5)

        # 4a. Analytic pi (no gradient needed)
        with torch.no_grad():
            pi_analytic = (mu_t - r) / (gamma * V)
            pi_analytic = pi_analytic.clamp(0, 1)
            W_analytic  = W_analytic * torch.exp(
                (r + pi_analytic*(mu_t - r) - 0.5*pi_analytic**2*V)*dt
                + pi_analytic*torch.sqrt(V)*dW_S
            )

        # 4b. ANN pi (gradient flows through this)
        state = torch.stack([
            torch.full((n_paths,), t*dt),
            V.detach(),
            W_ann.detach(),
            mu_t.detach()
        ], dim=1)
        pi_ann = policy(state).squeeze()
        W_ann = W_ann * torch.exp(
            (r + pi_ann*(mu_t - r) - 0.5*pi_ann**2*V)*dt
            + pi_ann*torch.sqrt(V)*dW_S   # same shock — fair comparison
        )

    # 5. Loss = negative expected utility of terminal wealth
    loss = -torch.mean(W_ann**(1 - gamma) / (1 - gamma))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        U_ann      = torch.mean(W_ann**(1-gamma)/(1-gamma)).item()
        U_analytic = torch.mean(W_analytic**(1-gamma)/(1-gamma)).item()
        print(f"Epoch {epoch} | ANN Utility: {U_ann:.4f} | Analytic Utility: {U_analytic:.4f}")